# 06 - Deployment and Scoring

This notebook scores live users with the trained production artifact.


In [ ]:
# Cell 0: Setup
from __future__ import annotations

from datetime import UTC, datetime
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from ml.pipelines.churn_common import create_db_engine, fetch_churn_feature_rows, build_feature_vector

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent.parent

model_path = ROOT / 'ml' / 'models' / 'churn_reorder_model.joblib'
assert model_path.exists(), 'Run notebook 03 first to train model.'

artifact = joblib.load(model_path)
model = artifact['model']
selected_features = artifact['selected_features']
threshold = float(artifact.get('threshold', 0.5))
print('Loaded model version:', artifact.get('model_version'))
print('Selected feature count:', len(selected_features))


In [ ]:
# Cell 1: Fetch live feature rows
engine = create_db_engine()
snapshot_ts = datetime.now(UTC)
rows = fetch_churn_feature_rows(
    engine,
    snapshot_ts=snapshot_ts,
    history_days=180,
    include_label=False,
)

assert rows, 'No eligible users to score.'
print('Rows fetched:', len(rows))


In [ ]:
# Cell 2: Score and segment risk
X = np.array([build_feature_vector(r, selected_features) for r in rows], dtype=float)
probs = model.predict_proba(X)[:, 1]

def risk_bucket(p: float) -> str:
    churn = 1.0 - p
    if churn >= 0.7:
        return 'high'
    if churn >= 0.4:
        return 'medium'
    return 'low'

scored = pd.DataFrame(
    {
        'user_id': [str(r['user_id']) for r in rows],
        'snapshot_ts': snapshot_ts.isoformat(),
        'reorder_probability_30d': probs,
        'churn_probability_30d': 1.0 - probs,
        'recommended_threshold': threshold,
        'predicted_positive': (probs >= threshold).astype(int),
    }
)
scored['risk_bucket'] = scored['reorder_probability_30d'].map(risk_bucket)

display(scored.sort_values('churn_probability_30d', ascending=False).head(25))


In [ ]:
# Cell 3: Save scored output
out_path = ROOT / 'ml' / 'data' / 'churn_scoring_latest.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
scored.to_csv(out_path, index=False)
print('Saved scoring file:', out_path)
print('Risk mix:')
print(scored['risk_bucket'].value_counts(normalize=True))


## Next Notebook

Proceed to `07_production_readiness.ipynb`.
